In [1]:
!mkdir -p /kaggle/working/src /kaggle/working/data/processed
!find /kaggle/input -name "*.py" -exec cp {} /kaggle/working/src/ \;
!find /kaggle/input -name "*.parquet" -exec cp {} /kaggle/working/data/processed/ \;
%cd /kaggle/working
!ls src data/processed
!nvidia-smi --query-gpu=name,memory.total --format=csv

/kaggle/working
data/processed:
test_heldout_attacks.parquet  test_seen.parquet       train_clean.parquet
test_reppen.parquet	      test_unseen.parquet     val.parquet
test_seen_attacks.parquet     train_attacked.parquet

src:
inspect_raid.py  metrics.py	     train_roberta.py
make_splits.py	 train_classical.py
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
!python src/train_roberta.py --seeds 42 --aug 0 --lrs 1e-5 2e-5 3e-5

Device: cuda
Tokenizing...
config.json: 100%|█████████████████████████████| 481/481 [00:00<00:00, 2.00MB/s]
tokenizer_config.json: 100%|█████████████████| 25.0/25.0 [00:00<00:00, 83.0kB/s]
vocab.json: 100%|████████████████████████████| 899k/899k [00:00<00:00, 16.2MB/s]
merges.txt: 100%|████████████████████████████| 456k/456k [00:00<00:00, 39.3MB/s]
tokenizer.json: 100%|██████████████████████| 1.36M/1.36M [00:00<00:00, 10.7MB/s]
  93,604 texts tokenized
Run aug=0.0 lr=1e-05 seed=42

model.safetensors: downloading bytes: ██████▉               |  156MB, 9.74MB/s  
model.safetensors: reconstructing file:  27%|█▌    |  134MB /  499MB            
model.safetensors: downloading bytes: ████████████▊         |  290MB, 23.7MB/s  
model.safetensors: reconstructing file:  67%|████  |  335MB /  499MB, 19.5MB/s  
model.safetensors: downloading bytes: ██████████████████████|  335MB, 31.5MB/s  
model.safetensors: reconstructing file: 100%|██████|  499MB /  499MB, 47.7MB/s  
Loading weights: 100%|█████

In [3]:
import pandas as pd
eff = pd.read_csv("results/roberta_efficiency.csv")
best_lr = eff.loc[eff["val_auroc"].idxmax(), "lr"]
print("Best learning rate (validation AUROC):", best_lr)
!python src/train_roberta.py --lrs {best_lr}

Best learning rate (validation AUROC): 2e-05
Device: cuda
Tokenizing...
  93,604 texts tokenized
skip aug=0.0 lr=2e-05 seed=42 (already done)
Run aug=0.0 lr=2e-05 seed=43
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2774.46it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the che